In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

In [ ]:
data_path = '/kaggle/input/rsna-breast-cancer-detection'

### Create the feature dataset with meta data features

In [ ]:
def make_dataset(is_train=True):
    # load metadata file
    if is_train:
        meta_df = pd.read_csv(f'{data_path}/train.csv')
    else:
        meta_df = pd.read_csv(f'{data_path}/test.csv')

    # replace NaN age with interger mean age
    meta_df['age'] = meta_df['age'].replace(to_replace=np.nan, value=int(meta_df['age'].mean()))

    # remove views with too few samples
    valid_views = {'MLO', 'CC'}
    is_valid_view = meta_df['view'].isin(valid_views)
    meta_df = meta_df.loc[is_valid_view, :]
    meta_df['view'] = meta_df['view'].map({'MLO': 0, 'CC': 1})
    
    # map laterality to integers
    meta_df['laterality'] = meta_df['laterality'].map({'L': 0, 'R': 1})
    
    # retain only relevant columns
    if is_train:
        meta_df = meta_df[['patient_id', 'age', 'laterality', 'cancer']]
    else:
        meta_df = meta_df[['patient_id', 'image_id', 'age', 'laterality', 'prediction_id']]

    # group by patient id
    meta_df = meta_df.groupby('patient_id').agg(list)

    # finalize df
    if is_train:
        meta_df = meta_df.explode(['age', 'laterality', 'cancer'])
        meta_df = meta_df.reset_index()
        meta_df = meta_df.drop(['patient_id'], axis=1)
        meta_df = meta_df.astype(int)
#         meta_df = meta_df.drop_duplicates()
    else:
        meta_df = meta_df.explode(['image_id', 'age', 'laterality', 'prediction_id'])
        meta_df = meta_df.reset_index()
        meta_df['image_path'] = meta_df[['patient_id', 'image_id']].apply(
            lambda x: f'{data_path}/test_images/{x[0]}/{x[1]}.dcm', axis=1
        )
        meta_df = meta_df.drop(['patient_id', 'image_id'], axis=1)
    
    return meta_df

In [ ]:
train_df = make_dataset(is_train=True)
test_df = make_dataset(is_train=False)

### NOTE: Here we would add the training image predictions to the train_df when we have them, as an additional feature.

In [ ]:
# load training set predictions
# train_df['img_pred'] = train_image_preds

train_df

In [ ]:
test_df

### NOTE: Here we will get the test set image predictions and add them as an additional feature

In [ ]:
# def load_dicom(image_path):
#     # load in dicom image
#     image_bytes = tf.io.read_file(image_path)
#     # load in image array
#     image = tfio.image.decode_dicom_image(
#         image_bytes, 
#         dtype=tf.uint16,
#         on_error='lossy')
    
#     return image[0]

# def pipeline(sample):
#     # load in dicom image
#     image_path = sample['image_path']
#     image = load_dicom(image_path)

#     return image

# # create tensorflow datasets
# test_ds = tf.data.Dataset.from_tensor_slices(dict(test_df))
# processed_test_ds = test_ds.map(pipeline, num_parallel_calls=2)

# def load_trained_model(path):
#     model = tf.keras.models.load_model(path)
#     return model

# model_path = '/kaggle/input/rsna-trained-models/best_model.h5'
# model = load_trained_model(model_path)

# batched_test_ds = processed_test_ds.batch(1, num_parallel_calls=tf.data.AUTOTUNE).prefetch(10)
# test_img_preds = model.predict(batched_test_ds)

# test_img_preds = np.squeeze(test_img_preds)

# test_df['img_pred'] = test_img_preds

### Here we will train the tabular classifier

In [ ]:
train_X = train_df.drop(['cancer'], axis=1).values
train_y = train_df['cancer'].values

test_X = test_df.drop(['prediction_id', 'image_path'], axis=1).values

In [ ]:
clf = LogisticRegression(max_iter=500)
clf.fit(train_X, train_y)

In [ ]:
test_pred = clf.predict_proba(test_X)
test_pred = test_pred[:, 1]

### Here we will find the cutoff resulting in maximum pfbeta for the training data

In [ ]:
def pfbeta_np(labels, preds, beta=1):
    preds = preds.clip(0, 1)
    y_true_count = labels.sum()
    ctp = preds[labels==1].sum()
    cfp = preds[labels==0].sum()
    beta_squared = beta * beta
    c_precision = ctp / (ctp + cfp)
    c_recall = ctp / y_true_count
    if (c_precision > 0 and c_recall > 0):
        result = (1 + beta_squared) * (c_precision * c_recall) / (beta_squared * c_precision + c_recall)
        return result
    else:
        return 0.0

In [ ]:
train_pred = clf.predict_proba(train_X)
train_pred = train_pred[:, 1]

In [ ]:
plt.hist(train_pred)

In [ ]:
all_pfbeta = []
for i in range(1, 100):
    cutoff = np.percentile(train_pred, i)
    train_pred_bin = (train_pred > cutoff).astype(float)
    pfbeta = pfbeta_np(train_y, train_pred_bin)
    all_pfbeta.append(pfbeta)
all_pfbeta = np.array(all_pfbeta)

In [ ]:
plt.plot(all_pfbeta)

In [ ]:
all_pfbeta.argmax()

In [ ]:
best_cutoff = np.percentile(train_pred, all_pfbeta.argmax())

In [ ]:
test_pred_bin = (test_pred > best_cutoff).astype(float)

In [ ]:
test_df['cancer'] = test_pred_bin
submission_df = test_df.groupby('prediction_id')['cancer'].agg(max)
submission_df = submission_df.reset_index()
submission_df.to_csv('submission.csv', index=False)

In [ ]:
submission_df